## Extracting Checkpoint, Bronze, Silver containers URLS

In [16]:
from common.spark_session import spark
from pyspark.sql.functions import col, current_timestamp

In [17]:
catalog = "sandbox-rey-01"
externalLocation = "loc_sandbox_sblakera"
schema_bronze = "bronze"
schema_silver = "silver"
raw_traffic_table = "raw_traffic"

In [18]:
checkpoint = spark.sql(f"""
    DESCRIBE EXTERNAL LOCATION `{externalLocation}`
""").select("url").collect()[0][0] + "checkpoints"

bronze = spark.sql(f"""
    DESCRIBE EXTERNAL LOCATION `{externalLocation}`
""").select("url").collect()[0][0] + "bronze"

silver = spark.sql(f"""
    DESCRIBE EXTERNAL LOCATION `{externalLocation}`
""").select("url").collect()[0][0] + "silver"

In [19]:
dfBronzeTraffic = (
    spark.readStream
        .table(f"`{catalog}`.`{schema_bronze}`.`{raw_traffic_table}`")
)

## DEDUP

In [20]:
def dedupDF(df):
    print('Deduplicating Dataframe')

    dfDedup = df.dropDuplicates()

    return dfDedup

## NULL REMOVAL

In [21]:
def handleNulls(df, column):
    print("Handling NULL values for string solumns", end='')
    df_string = df.fillna('Unknown', subset=column)
    print("Successfully handled string")

    print('Replacing Null values on Numberic Columns', end='')
    df_clean = df_string.fillna(0, subset=column)
    print("Successfully handled numeric value")

    return df_clean

## ELECTRIC VEHICLE COUNT

In [22]:
def evCount(df):
    dfEV = df.withColumn('Electric_Vehicles_Count', col('EV_Car') + col('EV_Bike'))
    print("Generated EV count")

    return dfEV

## Motor Vehicle Count

In [23]:
def motorVehicleCount(df):
    dfMotor = df.withColumn('Motor_Vehicles_Count', col('Two_wheeled_motor_vehicles') + col('Cars_and_taxis') + col('Buses_and_coaches') + col('LGV_Type') + col('HGV_Type') + col('Electric_Vehicles_Count'))
    print("Generated Motor Vehicle count")

    return dfMotor

## TRANSFORMED TIME

In [24]:
def createTransformedTime(df):
    print('Creating Transformed Time column : ',end='')
    df_timestamp = df.withColumn('Transformed_Time',
                      current_timestamp()
                      )
    print('Success!!')
    return df_timestamp

## WRITE SILVER TRAFFIC TABLE

In [25]:
def writeTrafficSilverTable(StreamingDF,catalog):
    print('Writing the silver_traffic Data : ',end='') 

    write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',checkpoint+ "/SilverTrafficLoad/Checkpt/")
                .outputMode('append')
                .queryName("SilverTrafficWriteStream")
                .trigger(availableNow=True)
                .toTable(f"`{catalog}`.`{schema_silver}`.`silver_traffic`"))
    
    write_StreamSilver.awaitTermination()
    print(f'Writing `{catalog}`.`{schema_silver}`.`silver_traffic` Success!')

## FUNCTION CALLING

In [26]:
dfBronzeTraffic.schema.names

['Record_ID',
 'Count_point_id',
 'Direction_of_travel',
 'Year',
 'Count_date',
 'hour',
 'Region_id',
 'Region_name',
 'Local_authority_name',
 'Road_name',
 'Road_Category_ID',
 'Start_junction_road_name',
 'End_junction_road_name',
 'Latitude',
 'Longitude',
 'Link_length_km',
 'Pedal_cycles',
 'Two_wheeled_motor_vehicles',
 'Cars_and_taxis',
 'Buses_and_coaches',
 'LGV_Type',
 'HGV_Type',
 'EV_Car',
 'EV_Bike',
 'Extract_Time']

In [27]:
# Remove duplicate rows
dfBronzeDedup = dedupDF(dfBronzeTraffic)

# Replace Null Values
allColumns = dfBronzeDedup.schema.names
dfBronzeNull = handleNulls(dfBronzeDedup, allColumns)

# Get total EV Count
dfEV = evCount(dfBronzeNull)

# Get total Motor Vehicle Count
dfMV = motorVehicleCount(dfEV)

# Add Transformed Time column
dfTransformed = createTransformedTime(dfMV)


Deduplicating Dataframe
Handling NULL values for string solumnsSuccessfully handled string
Replacing Null values on Numberic ColumnsSuccessfully handled numeric value
Generated EV count
Generated Motor Vehicle count
Creating Transformed Time column : Success!!


In [28]:
# Write to Silver Traffic table

writeTrafficSilverTable(dfTransformed, catalog)

Writing the silver_traffic Data : Writing `sandbox-rey-01`.`silver`.`silver_traffic` Success!


In [13]:
dfStaticTraffic = spark.read.table(f"`{catalog}`.`{schema_bronze}`.`{raw_traffic_table}`")
dfStaticMV = motorVehicleCount(evCount(handleNulls(dedupDF(dfStaticTraffic), dfStaticTraffic.schema.names)))
dfStaticMV.createOrReplaceTempView("ev_preview")

display(spark.sql("SELECT * FROM ev_preview"))

Deduplicating Dataframe
Handling NULL values for string solumnsSuccessfully handled string
Replacing Null values on Numberic ColumnsSuccessfully handled numeric value
Generated EV count
Generated Motor Vehicle count


,Record_ID,Count_point_id,Direction_of_travel,Year,Count_date,hour,Region_id,Region_name,Local_authority_name,Road_name,Road_Category_ID,Start_junction_road_name,End_junction_road_name,Latitude,Longitude,Link_length_km,Pedal_cycles,Two_wheeled_motor_vehicles,Cars_and_taxis,Buses_and_coaches,LGV_Type,HGV_Type,EV_Car,EV_Bike,Extract_Time,Electric_Vehicles_Count,Motor_Vehicles_Count
0,18561,37072,N,2014,10/24/2014 0:00,7,1,South West,Devon,A380,4,A381,A381/A383,50.536947,-3.587026,2.5,0,20,1069,6,383,21,8,9,2026-05-08 01:19:08.016,17,1516
1,18599,17838,S,2014,5/15/2014 0:00,9,10,West Midlands,Solihull,M42,1,4,5,52.397082,-1.763342,3.9,0,10,3297,17,479,103,22,48,2026-05-08 01:19:08.016,70,3976
2,18601,17838,S,2014,5/15/2014 0:00,11,10,West Midlands,Solihull,M42,1,4,5,52.397082,-1.763342,3.9,0,8,2696,16,513,106,21,36,2026-05-08 01:19:08.016,57,3396
3,18625,7903,W,2014,6/9/2014 0:00,11,7,East of England,Hertfordshire,M25,1,20,21,51.714066,-0.409206,4.7,0,10,3383,26,633,115,35,30,2026-05-08 01:19:08.016,65,4232
4,18627,7903,E,2014,6/9/2014 0:00,13,7,East of England,Hertfordshire,M25,1,20,21,51.714066,-0.409206,4.7,0,13,3097,10,820,184,44,47,2026-05-08 01:19:08.016,91,4215
5,18634,16073,N,2014,10/21/2014 0:00,8,7,East of England,Central Bedfordshire,A1,3,End of A1M,A6001 Holme Green,52.065000,-0.238332,6.8,0,4,1125,3,177,35,5,11,2026-05-08 01:19:08.016,16,1360
6,18651,37625,W,2014,9/10/2014 0:00,8,7,East of England,Suffolk,A1302,4,A134,A143,52.235343,0.722436,1.3,36,14,727,8,53,7,4,0,2026-05-08 01:19:08.016,4,813
7,18652,37625,W,2014,9/10/2014 0:00,9,7,East of England,Suffolk,A1302,4,A134,A143,52.235343,0.722436,1.3,4,6,505,5,39,3,3,0,2026-05-08 01:19:08.016,3,561
8,18654,37625,W,2014,9/10/2014 0:00,11,7,East of England,Suffolk,A1302,4,A134,A143,52.235343,0.722436,1.3,5,3,407,2,41,10,5,2,2026-05-08 01:19:08.016,7,470
9,18668,36398,S,2014,9/30/2014 0:00,13,10,West Midlands,Staffordshire,A38,4,A5,A5148,52.654727,-1.809851,2.8,0,5,739,2,137,26,7,15,2026-05-08 01:19:08.016,22,931
